In [1]:
from scipy.stats import qmc
import numpy as np
import pandas as pd

from scipy.stats import norm
from scipy.optimize import minimize
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.gaussian_process.kernels import Matern

In [2]:
#Function 5
#Extracting updated data and turning it into a pandas dataframe
data = pd.read_csv('Data/Week 6 - Function 5.csv') 
columns = ['Input 1', 'Input 2','Input 3', 'Input 4','Outputs']
#Remove columns with NaN values
data = data.dropna(axis = 1)
data.columns = columns
#Add Week 7's data to our pandas dataframe
new_data7 = np.array([0.992528, 0.832966, 0.994330, 0.998665,
                     6089.295977690454])
data.loc[len(data)] = new_data7
#Add Week 8's data to our pandas dataframe
new_data8 = np.array([0.970115, 0.990124, 0.944969, 0.988218,
                     6822.940246057326])
data.loc[len(data)] = new_data8
#Add Week 10's data to our pandas dataframe
new_data10 = np.array([0.992917, 0.951906, 0.976171, 0.984828,
                     7013.134740405846])
data.loc[len(data)] = new_data10
#Add Week 11's data to our pandas dataframe
new_data11 = np.array([0.954213, 0.952188, 0.932887, 0.952041,
                     5331.597987100769])
data.loc[len(data)] = new_data11
data

,Input 1,Input 2,Input 3,Input 4,Outputs
0,0.191447,0.038193,0.607418,0.414584,64.443440
1,0.758653,0.536518,0.656000,0.360342,18.301380
2,0.438350,0.804340,0.210245,0.151295,0.112940
3,0.706051,0.534192,0.264243,0.482088,4.210898
4,0.836478,0.193610,0.663893,0.785649,258.370500
5,0.683432,0.118663,0.829046,0.567577,78.434390
6,0.553621,0.667350,0.323806,0.814870,57.571540
7,0.352356,0.322242,0.116979,0.473113,109.571900
8,0.153786,0.729382,0.422598,0.443074,8.847992
9,0.463442,0.630025,0.107906,0.957644,233.223600


In [4]:
#Extract the Data Into Numpy Arrays
X = np.array(data[['Input 1', 'Input 2', 'Input 3', 'Input 4']])
Y = np.array(data[['Outputs']])


In [66]:
#Using latin hypercube sampling to initialise a set of candidate points in a trust region
#Initialise points in the unit hypercube
d = 4
n = 40000
sampler = qmc.LatinHypercube(d, seed = 42)
grid = sampler.random(n) 

#Scale points generated in the unit hypercube to be inside trust region
#Trust hypercube of radius delta centered at best point with from observed data is 
#[x_best-delta, x_best_plus]
x_best = X[np.argmax(Y)]
#Set radius of hypercube
delta = 0.005
#note that delta is set very low because best point is very close to the boundary of domain
#lower and upper bounds of hypercube

lb = x_best-40*delta 
ub = x_best+delta

#scale points from unit hypercube to be inside trust region
scaled_grid = np.clip(lb + grid * (ub-lb),0,1)


In [67]:
#We now set the Kernel to be the Matern Kernel
nu = 2.5
kernel = Matern(
    length_scale_bounds=(1e-2, 2),
    nu= nu
)
print("Kernel:", kernel)
gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-10,
    normalize_y = True)
print(gp)
gp.fit(X,Y)

mu, std = gp.predict(scaled_grid, return_std = True)

Kernel: Matern(length_scale=1, nu=2.5)
GaussianProcessRegressor(kernel=Matern(length_scale=1, nu=2.5),
                         normalize_y=True)


In [68]:
#Calculate UCB
beta = 0.2 #set exploration parameter
UCB = mu + beta*std 
#Find point that maximises UCB acquisition function
x_next = scaled_grid[np.argmax(UCB)]
print("Next suggested point:", np.round(x_next,6))


Next suggested point: [0.991791 0.954881 0.96989  0.987413]


In [3]:
#Week 12

#Add new data
new_data = np.array([0.991791, 0.954881, 0.969890, 0.987413,6981.275294969248])
data.loc[len(data)] = new_data
data

,Input 1,Input 2,Input 3,Input 4,Outputs
0,0.191447,0.038193,0.607418,0.414584,64.443440
1,0.758653,0.536518,0.656000,0.360342,18.301380
2,0.438350,0.804340,0.210245,0.151295,0.112940
3,0.706051,0.534192,0.264243,0.482088,4.210898
4,0.836478,0.193610,0.663893,0.785649,258.370500
5,0.683432,0.118663,0.829046,0.567577,78.434390
6,0.553621,0.667350,0.323806,0.814870,57.571540
7,0.352356,0.322242,0.116979,0.473113,109.571900
8,0.153786,0.729382,0.422598,0.443074,8.847992
9,0.463442,0.630025,0.107906,0.957644,233.223600


In [4]:
#Extract the Data Into Numpy Arrays
X = np.array(data[['Input 1', 'Input 2', 'Input 3', 'Input 4']])
Y = np.array(data[['Outputs']])

In [9]:
#Using latin hypercube sampling to initialise a set of candidate points in a trust region
#Initialise points in the unit hypercube
d = 4
n = 40001
sampler = qmc.LatinHypercube(d, seed = 42)
grid = sampler.random(n) 

#Scale points generated in the unit hypercube to be inside trust region
#Trust hypercube of radius delta centered at best point with from observed data is 
#[x_best-delta, x_best_plus]
x_best = X[np.argmax(Y)]
#Reduce radius of hypercube by 10% since there was no improvement
delta = 0.005*0.9
#note that delta is set very low because best point is very close to the boundary of domain
#lower and upper bounds of hypercube

lb = x_best-40*delta 
ub = x_best+delta

#scale points from unit hypercube to be inside trust region
scaled_grid = np.clip(lb + grid * (ub-lb),0,1)
print("Center of hypercube:", x_best)
print("Upper bound", ub)
print("lower bound", lb)

Center of hypercube: [0.992917 0.951906 0.976171 0.984828]
Upper bound [0.997417 0.956406 0.980671 0.989328]
lower bound [0.812917 0.771906 0.796171 0.804828]


In [10]:
#We now set the Kernel to be the Matern Kernel
nu = 2.5
kernel = Matern(
    length_scale_bounds=(1e-2, 2),
    nu= nu
)
print("Kernel:", kernel)
gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-10,
    normalize_y = True)
print(gp)
gp.fit(X,Y)

mu, std = gp.predict(scaled_grid, return_std = True)

Kernel: Matern(length_scale=1, nu=2.5)
GaussianProcessRegressor(kernel=Matern(length_scale=1, nu=2.5),
                         normalize_y=True)


In [11]:
#Calculate UCB
beta = 0.2 #set exploration parameter
UCB = mu + beta*std 
#Find point that maximises UCB acquisition function
x_next = scaled_grid[np.argmax(UCB)]
print("Next suggested point:", np.round(x_next,6))


Next suggested point: [0.99611  0.947533 0.976275 0.986529]


In [4]:
#Week 13

#Add new data
new_data = np.array([0.996110, 0.947533, 0.976275, 0.986529,7032.547252121981])
data.loc[len(data)] = new_data
data

,Input 1,Input 2,Input 3,Input 4,Outputs
0,0.191447,0.038193,0.607418,0.414584,64.443440
1,0.758653,0.536518,0.656000,0.360342,18.301380
2,0.438350,0.804340,0.210245,0.151295,0.112940
3,0.706051,0.534192,0.264243,0.482088,4.210898
4,0.836478,0.193610,0.663893,0.785649,258.370500
5,0.683432,0.118663,0.829046,0.567577,78.434390
6,0.553621,0.667350,0.323806,0.814870,57.571540
7,0.352356,0.322242,0.116979,0.473113,109.571900
8,0.153786,0.729382,0.422598,0.443074,8.847992
9,0.463442,0.630025,0.107906,0.957644,233.223600


In [5]:
#Extract the Data Into Numpy Arrays
X = np.array(data[['Input 1', 'Input 2', 'Input 3', 'Input 4']])
Y = np.array(data[['Outputs']])

In [7]:
#Using latin hypercube sampling to initialise a set of candidate points in a trust region
#Initialise points in the unit hypercube
d = 4
n = 40001
sampler = qmc.LatinHypercube(d, seed = 42)
grid = sampler.random(n) 

#Scale points generated in the unit hypercube to be inside trust region
#Trust hypercube of radius delta centered at best point with from observed data is 
#[x_best-delta, x_best_plus]
x_best = X[np.argmax(Y)]
#Increase radius of hypercube by 10% since there was an improvement
delta = 0.005*0.9*1.1
#note that delta is set very low because best point is very close to the boundary of domain
#lower and upper bounds of hypercube

lb = x_best-40*delta 
ub = x_best+delta

#scale points from unit hypercube to be inside trust region
scaled_grid = np.clip(lb + grid * (ub-lb),0,1)
print("Center of hypercube:", x_best)
print("Upper bound", ub)
print("lower bound", lb)

Center of hypercube: [0.99611  0.947533 0.976275 0.986529]
Upper bound [1.00106  0.952483 0.981225 0.991479]
lower bound [0.79811  0.749533 0.778275 0.788529]


In [10]:
#We now set the Kernel to be the Matern Kernel
nu = 2.5
kernel = Matern(
    length_scale_bounds=(1e-2, 2),
    nu= nu
)
print("Kernel:", kernel)
gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-10,
    normalize_y = True)
print(gp)
gp.fit(X,Y)

mu, std = gp.predict(scaled_grid, return_std = True)

Kernel: Matern(length_scale=1, nu=2.5)
GaussianProcessRegressor(kernel=Matern(length_scale=1, nu=2.5),
                         normalize_y=True)


In [13]:
#Calculate UCB
beta = 0.2 #set exploration parameter
UCB = mu + beta*std 
#Find point that maximises UCB acquisition function
x_next = scaled_grid[np.argmax(UCB)]
print("Next suggested point:", np.round(x_next,6))

Next suggested point: [0.999623 0.942722 0.976389 0.9884  ]
